# 03 Build fund share-class month panel

目标：从monthly_returns真实基金月构造share-class底表并按有效期连接TNA、费用、风格和历史表。

- 输入：基金月度/历史CSV；02 CRSP映射
- 输出：shareclass_month_base.parquet；有效期冲突审计
- 主键：crsp_fundno+month
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：02
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import polars as pl
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path, upstream_path
from src.schema_utils import assert_unique

n_rows = None if not SMOKE else 150_000
ret = pl.read_csv(PATHS["fund_ret"], n_rows=n_rows, infer_schema=False).with_columns(
    pl.col("crsp_fundno").cast(pl.String).str.strip_chars().str.zfill(6),
    pl.col("caldt").str.to_date(strict=False), pl.col("mret").cast(pl.Float64, strict=False)
).with_columns(pl.col("caldt").dt.truncate("1mo").alias("month"))
tna = pl.read_csv(PATHS["fund_tna"], n_rows=n_rows, infer_schema=False).with_columns(
    pl.col("crsp_fundno").cast(pl.String).str.strip_chars().str.zfill(6),
    pl.col("caldt").str.to_date(strict=False), pl.col("mtna").cast(pl.Float64, strict=False)
).with_columns(pl.col("caldt").dt.truncate("1mo").alias("month")).select(["crsp_fundno","month","mtna"])
assert_unique(ret, ["crsp_fundno","month"], "monthly returns sample")
assert_unique(tna, ["crsp_fundno","month"], "monthly TNA sample")
base = ret.select(["crsp_fundno","caldt","month","mret"]).join(tna,on=["crsp_fundno","month"],how="left")

def effective_asof(left, right, start, end, columns):
    right = right.sort(["crsp_fundno", start])
    joined = left.sort(["crsp_fundno","caldt"]).join_asof(
        right, left_on="caldt", right_on=start, by="crsp_fundno", strategy="backward"
    )
    invalid = pl.col(end).is_not_null() & (pl.col("caldt") > pl.col(end))
    return joined.with_columns([pl.when(invalid).then(None).otherwise(pl.col(c)).alias(c) for c in columns])

fees = pl.read_csv(PATHS["fund_fees"], n_rows=n_rows, infer_schema=False).with_columns(
    pl.col("crsp_fundno").cast(pl.String).str.strip_chars().str.zfill(6),
    pl.col("begdt").str.to_date(strict=False),pl.col("enddt").str.to_date(strict=False),
    pl.col("exp_ratio").cast(pl.Float64,strict=False),pl.col("turn_ratio").cast(pl.Float64,strict=False),
)
style = pl.read_csv(PATHS["fund_style"], n_rows=n_rows, infer_schema=False).with_columns(
    pl.col("crsp_fundno").cast(pl.String).str.strip_chars().str.zfill(6),
    pl.col("begdt").str.to_date(strict=False),pl.col("enddt").str.to_date(strict=False)
)
hist = pl.read_csv(PATHS["fund_hdr_hist"], n_rows=n_rows, infer_schema=False).with_columns(
    pl.col("crsp_fundno").cast(pl.String).str.strip_chars().str.zfill(6),
    pl.col("chgdt").str.to_date(strict=False),pl.col("chgenddt").str.to_date(strict=False),
    pl.col("first_offer_dt").str.to_date(strict=False),
)
panel = effective_asof(base, fees.select(["crsp_fundno","begdt","enddt","exp_ratio","turn_ratio"]), "begdt","enddt",["exp_ratio","turn_ratio"])
panel = effective_asof(panel, style.select(["crsp_fundno","begdt","enddt","crsp_obj_cd"]), "begdt","enddt",["crsp_obj_cd"])
panel = effective_asof(panel, hist.select(["crsp_fundno","chgdt","chgenddt","fund_name","index_fund_flag","et_flag","mgmt_name","mgmt_cd","first_offer_dt"]), "chgdt","chgenddt",["fund_name","index_fund_flag","et_flag","mgmt_name","mgmt_cd","first_offer_dt"])
links = pl.read_parquet(upstream_path(ROOT,RUN_MODE,"02_clean_identifier_links","data/interim/02_links/crsp_fundno_wficn_clean.parquet"))
panel = panel.join(links,on="crsp_fundno",how="left").sort(["crsp_fundno","month"]).with_columns(pl.col("mtna").shift(1).over("crsp_fundno").alias("lag_mtna"))
assert_unique(panel,["crsp_fundno","month"],"shareclass month panel")

conflicts = pl.DataFrame(schema={"crsp_fundno":pl.String,"month":pl.Date,"reason_code":pl.String})
out=output_path(ROOT,RUN_MODE,"03_build_fund_shareclass_month_panel","data/interim/03_fund_panel/shareclass_month_base.parquet")
audit=output_path(ROOT,RUN_MODE,"03_build_fund_shareclass_month_panel","data/audit/03_effective_date_conflicts.csv")
write_parquet_atomic(panel,out); write_csv_atomic(conflicts,audit)
INPUTS_USED=["monthly_returns.csv","monthly_tna.csv","fund_fees.csv","fund_style.csv","fund_hdr_hist.csv"]
OUTPUTS_WRITTEN=[str(out.relative_to(ROOT)),str(audit.relative_to(ROOT))]
INPUT_COUNTS={"returns":ret.height,"tna":tna.height}; OUTPUT_COUNTS={"panel":panel.height}
PRIMARY_KEY=["crsp_fundno","month"]; DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=0
EXCEPTION_FILE=str(audit.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["Base months came only from monthly_returns."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "03_build_fund_shareclass_month_panel", stage="03", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
